# Table S62 – Accuracy & ARI Analysis (Two-Step Baseline: HC)

Reads `.Rds` simulation outputs from the **two-step baseline** pipeline and produces

```
output/
  tabS62/
    baseline/
      twostep_seed*_c*_frailty*_censor*.Rds
    misspec_frailty/
      ...
```

In [8]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import rdata
from pathlib import Path
from sklearn.metrics import adjusted_rand_score

In [9]:
# ── Configuration ──────────────────────────────────────────────────────────
BASE_DIR = Path("output/tabS62")

SCENARIOS = [
    "baseline",
    "misspec_frailty",
    "misspec_baseline",
    "misspec_both",
    "weak_separation",
    "medium_separation",
    "imbalanced_moderate",
    "imbalanced_severe",
    "worst_case",
]

SCENARIO_LABELS = {
    "baseline":            r"(a) Baseline",
    "imbalanced_moderate": r"(b) Moderate imbalance",
    "imbalanced_severe":   r"(c) Severe imbalance",
    "medium_separation":   r"(d) Medium separation",
    "weak_separation":     r"(e) Low separation",
    "misspec_frailty":     r"(f) Frailty misspec",
    "misspec_baseline":    r"(g) Baseline misspec",
    "misspec_both":        r"(h) Joint misspec",
    "worst_case":          r"(i) Worst-case",
}

METHODS = ["kmeans", "hc_sin", "hc_ward"]   # keys that match clusters_* fields in .Rds
METHOD_LABELS = {"kmeans": "K-Means", "hc_sin": "Hierarchical (Single)", "hc_ward": "Hierarchical (Ward)"}

In [10]:
# ── Helper: greedy relabelling (predicted → true) ──────────────────────────
def simple_relabel(true_labels, pred_labels):
    """Map predicted cluster ids to true ids by maximum overlap (greedy)."""
    true_labels = pd.Series(true_labels)
    pred_labels = pd.Series(pred_labels)

    mapping = {}
    available_true = set(true_labels.unique())

    for p in pred_labels.unique():
        overlaps = {
            t: ((true_labels == t) & (pred_labels == p)).sum()
            for t in available_true
        }
        best = max(overlaps, key=overlaps.get)
        mapping[p] = best
        available_true.discard(best)

    return pred_labels.map(mapping)

In [4]:
# ── Load & process all .Rds files ─────────────────────────────────────────
#
# Each file is an R list saved with saveRDS().
#
# Expected fields (from twostep_baseline.R):
#   seed, scenario, c,
#   true                  – true cluster labels (length N)
#   clusters_kmeans       – k-means labels      (length N)
#   clusters_hc           – hierarchical labels (length N)
#   clusters_spectral     – spectral labels     (length N, may contain NA)
#   dgp_frailty, dgp_baseline, fit_frailty, fit_baseline,
#   separation, balance, censoring_rate
#
# Censoring type is encoded in the filename: _censoradministrative_ / _censornormal_

# results[(scenario, censor_name, method)] = {"acc": [], "ari": []}
results = {}
skipped = 0
total   = 0

for scenario in SCENARIOS:
    folder = BASE_DIR / scenario
    if not folder.exists():
        print(f"[INFO] Folder not found, skipping: {folder}")
        continue

    rds_files = sorted(folder.glob("*.Rds"))
    print(f"  {scenario}: {len(rds_files)} files")
    total += len(rds_files)

    for file in rds_files:
        fname = file.name

        # ── Determine censoring from filename ──────────────────────────
        if "censoradministrative" in fname:
            censor_name = "Administrative"
        elif "censornormal" in fname:
            censor_name = "Normal"
        else:
            skipped += 1
            continue

        if "frailty_intensity0.5" in fname:
            frailty_intensity = 0.5
        elif "frailty_intensity1.5" in fname:
            frailty_intensity = 1.5
        elif "frailty_intensity1" in fname:
            frailty_intensity = 1

        # ── Read the Rds file ──────────────────────────────────────────
        try:
            obj = rdata.read_rds(file)
            obj = {str(k): v for k, v in obj.items()}
        except Exception as e:
            print(f"  [WARNING] Could not read {fname}: {e}")
            skipped += 1
            continue

        # ── True labels ────────────────────────────────────────────────
        true_labels = np.asarray(obj["true"]).ravel().astype(int)

        # ── Evaluate each clustering method ────────────────────────────
        for method in METHODS:
            rds_key = f"clusters_{method}"
            pred_raw = np.asarray(obj.get(rds_key, [])).ravel()

            # Skip if all NA or empty
            if len(pred_raw) == 0 or np.all(np.isnan(pred_raw.astype(float))):
                continue

            # Remove any NA positions
            valid = ~np.isnan(pred_raw.astype(float))
            pred_labels = pred_raw[valid].astype(int)
            true_sub    = true_labels[valid]

            aligned  = np.asarray(simple_relabel(true_sub, pred_labels))
            accuracy = (true_sub == aligned).mean()
            ari      = adjusted_rand_score(true_sub, pred_labels)

            key = (scenario, frailty_intensity, method)
            if key not in results:
                results[key] = {"acc": [], "ari": []}
            results[key]["acc"].append(accuracy)
            results[key]["ari"].append(ari)

print(f"\nTotal files found: {total}  |  skipped: {skipped}")

  baseline: 300 files
  misspec_frailty: 300 files
  misspec_baseline: 300 files
  misspec_both: 300 files
  weak_separation: 300 files
  medium_separation: 300 files
  imbalanced_moderate: 300 files
  imbalanced_severe: 300 files
  worst_case: 300 files

Total files found: 2700  |  skipped: 0


In [11]:
# ── Build summary DataFrame ────────────────────────────────────────────────
rows = []
for (scenario, frailty_intensity, method), vals in results.items():
    acc = np.array(vals["acc"])
    ari = np.array(vals["ari"])
    rows.append({
        "Scenario"  : scenario,
        "frailty_intensity": frailty_intensity,
        "Method"    : method,
        "n_reps"    : len(acc),
        "Acc_mean"  : np.round(acc.mean(),        3),
        "Acc_median": np.round(np.median(acc),    3),
        "Acc_sd"    : np.round(acc.std(),         3),
        "ARI_mean"  : np.round(ari.mean(),        3),
        "ARI_median": np.round(np.median(ari),    3),
        "ARI_sd"    : np.round(ari.std(),         3)
    })

scenario_order = {s: i for i, s in enumerate(SCENARIOS)}
method_order   = {m: i for i, m in enumerate(METHODS)}

summary_df = (
    pd.DataFrame(rows)
    .assign(
        scen_ord = lambda df: df["Scenario"].map(scenario_order),
        meth_ord = lambda df: df["Method"].map(method_order),
    )
    .sort_values(["scen_ord", "frailty_intensity", "meth_ord"])
    .drop(columns=["scen_ord", "meth_ord"])
    .reset_index(drop=True)
)

print(summary_df.to_string())

               Scenario  frailty_intensity  Method  n_reps  Acc_mean  Acc_median  Acc_sd  ARI_mean  ARI_median  ARI_sd
0              baseline                0.5  hc_sin     100     0.708       0.680   0.183     0.585       0.568   0.277
1              baseline                1.0  hc_sin     100     0.751       0.688   0.200     0.649       0.573   0.291
2              baseline                1.5  hc_sin     100     0.689       0.678   0.200     0.558       0.559   0.305
3       misspec_frailty                0.5  hc_sin     100     0.725       0.680   0.197     0.611       0.571   0.292
4       misspec_frailty                1.0  hc_sin     100     0.725       0.680   0.197     0.611       0.571   0.292
5       misspec_frailty                1.5  hc_sin     100     0.725       0.680   0.197     0.611       0.571   0.292
6      misspec_baseline                0.5  hc_sin     100     0.732       0.681   0.186     0.621       0.570   0.277
7      misspec_baseline                1.0  hc_s

In [12]:
# ── Identify best method per (Scenario, frailty_intensity) block ──────────
#
# "Best" = highest Acc_mean  (ties broken by lowest Acc_sd).
# Applied independently within each (Scenario, frailty_intensity) group.

def find_bold_rows(group):
    max_acc  = group["Acc_mean"].max()
    cands    = group[group["Acc_mean"] == max_acc]
    min_sd   = cands["Acc_sd"].min()
    return (group["Acc_mean"] == max_acc) & (group["Acc_sd"] == min_sd)

bold_mask = pd.Series(False, index=summary_df.index)
for (scen, fi), grp in summary_df.groupby(["Scenario", "frailty_intensity"], sort=False):
    bold_mask.loc[grp.index] = find_bold_rows(grp)

summary_df["bold"] = bold_mask
print("Bold (best) rows per scenario x frailty_intensity:")
print(summary_df[summary_df["bold"]][["Scenario","frailty_intensity","Method","Acc_mean","ARI_mean"]])


Bold (best) rows per scenario x frailty_intensity:
               Scenario  frailty_intensity  Method  Acc_mean  ARI_mean
0              baseline                0.5  hc_sin     0.708     0.585
1              baseline                1.0  hc_sin     0.751     0.649
2              baseline                1.5  hc_sin     0.689     0.558
3       misspec_frailty                0.5  hc_sin     0.725     0.611
4       misspec_frailty                1.0  hc_sin     0.725     0.611
5       misspec_frailty                1.5  hc_sin     0.725     0.611
6      misspec_baseline                0.5  hc_sin     0.732     0.621
7      misspec_baseline                1.0  hc_sin     0.755     0.657
8      misspec_baseline                1.5  hc_sin     0.717     0.599
9          misspec_both                0.5  hc_sin     0.679     0.533
10         misspec_both                1.0  hc_sin     0.679     0.533
11         misspec_both                1.5  hc_sin     0.679     0.533
12      weak_separation   

In [13]:
# ── New simplified LaTeX table: Administrative censoring, HC-Single only, C=3 ──
#
# Filters summary_df down to:
#   - method == "hc_sin"   (hierarchical clustering, single linkage)
#   - (this dataset is administrative censoring only / superimposed C=3 by construction;
#      add explicit filters below if those columns exist in your df)
#
# Produces a table with columns: Scenario | theta | Acc(Mean,Median,SD) | ARI(Mean,Median,SD)

SCENARIO_LABELS_THETA = {
    "baseline"           : "Baseline",
    "misspec_frailty"    : r"Misspec.\ Frailty",
    "misspec_baseline"   : r"Misspec.\ Baseline",
    "misspec_both"       : r"Misspec.\ Both",
    "weak_separation"    : "Weak Sep.",
    "medium_separation"  : "Medium Sep.",
    "imbalanced_moderate": r"Imbal.\ Moderate",
    "imbalanced_severe"  : r"Imbal.\ Severe",
    "worst_case"         : "Worst Case",
}

def fmt_val_simple(v):
    return f"{v:.3f}"

def build_latex_table_theta(df, method="hc_sin"):
    # Filter to the single method of interest (HC single linkage).
    # If your df also has explicit "censoring" or "n_clusters_superimposed" columns,
    # add `& (df["censoring"] == "Administrative") & (df["n_clusters"] == 3)` etc. here.
    fdf_all = df[df["Method"] == method].copy()

    lines = []
    lines.append(
        r"""\begin{table}[!htbp]
\centering
\footnotesize
\begin{tabular}{ll ccc ccc}
\toprule
\textbf{Scenario} & $\theta$
    & \multicolumn{3}{c}{\textbf{Accuracy}}
    & \multicolumn{3}{c}{\textbf{ARI}} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-8}
 & & Mean & Median & SD & Mean & Median & SD \\
\toprule"""
    )

    scenarios = [s for s in SCENARIOS if s in fdf_all["Scenario"].unique()]

    for si, scenario in enumerate(scenarios):
        sdf = fdf_all[fdf_all["Scenario"] == scenario].sort_values("frailty_intensity")
        n_rows_scenario = len(sdf)
        scen_label = SCENARIO_LABELS_THETA.get(scenario, scenario)

        if si > 0:
            lines.append(r"\midrule")

        for ri, (_, row) in enumerate(sdf.iterrows()):
            theta_label = f"{row['frailty_intensity']:g}"

            if ri == 0:
                col1 = rf"\multirow{{{n_rows_scenario}}}{{*}}{{{scen_label}}}"
            else:
                col1 = ""

            data_cols = " & ".join([
                fmt_val_simple(row["Acc_mean"]),
                fmt_val_simple(row["Acc_median"]),
                fmt_val_simple(row["Acc_sd"]),
                fmt_val_simple(row["ARI_mean"]),
                fmt_val_simple(row["ARI_median"]),
                fmt_val_simple(row["ARI_sd"]),
            ])

            line = f"  {col1} & {theta_label} & {data_cols}\\\\"
            lines.append(line)

    lines.append(
        r"""\bottomrule
\end{tabular}
\caption{\small Empirical means, medians, and standard deviations of \textit{accuracy} and \textit{ARI} across $B=100$ simulation replicates, evaluated for administrative censoring and $\theta \in \{0.5, 1, 1.5\}$ with $C=3$ true clusters, using a two-stage approach with hierarchical clustering (single linkage), superimposing a number of clusters equal to $C=3$. Performance metrics are computed over these runs only.}
\label{tab:newsim_comparison}
\end{table}"""
    )

    return "\n".join(lines)


latex_str_theta = build_latex_table_theta(summary_df, method="hc_sin")
print(latex_str_theta)


\begin{table}[!htbp]
\centering
\footnotesize
\begin{tabular}{ll ccc ccc}
\toprule
\textbf{Scenario} & $\theta$
    & \multicolumn{3}{c}{\textbf{Accuracy}}
    & \multicolumn{3}{c}{\textbf{ARI}} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-8}
 & & Mean & Median & SD & Mean & Median & SD \\
\toprule
  \multirow{3}{*}{Baseline} & 0.5 & 0.708 & 0.680 & 0.183 & 0.585 & 0.568 & 0.277\\
   & 1 & 0.751 & 0.688 & 0.200 & 0.649 & 0.573 & 0.291\\
   & 1.5 & 0.689 & 0.678 & 0.200 & 0.558 & 0.559 & 0.305\\
\midrule
  \multirow{3}{*}{Misspec.\ Frailty} & 0.5 & 0.725 & 0.680 & 0.197 & 0.611 & 0.571 & 0.292\\
   & 1 & 0.725 & 0.680 & 0.197 & 0.611 & 0.571 & 0.292\\
   & 1.5 & 0.725 & 0.680 & 0.197 & 0.611 & 0.571 & 0.292\\
\midrule
  \multirow{3}{*}{Misspec.\ Baseline} & 0.5 & 0.732 & 0.681 & 0.186 & 0.621 & 0.570 & 0.277\\
   & 1 & 0.755 & 0.686 & 0.185 & 0.657 & 0.569 & 0.266\\
   & 1.5 & 0.717 & 0.683 & 0.176 & 0.599 & 0.568 & 0.265\\
\midrule
  \multirow{3}{*}{Misspec.\ Both} & 0.5 & 0.679 & 0.680 & 0.1

In [ ]:
# ── Save new theta-style LaTeX table to file ──────────────────────────────
out_path_theta = Path("TabS62_AccuracyARI_theta.tex")
out_path_theta.write_text(latex_str_theta)
print(f"LaTeX table written to {out_path_theta.resolve()}")

In [122]:
# ── LaTeX table generator ──────────────────────────────────────────────────

SCENARIO_LABELS = {
    "baseline"           : "Baseline",
    "misspec_frailty"    : r"Misspec.\ Frailty",
    "misspec_baseline"   : r"Misspec.\ Baseline",
    "misspec_both"       : r"Misspec.\ Both",
    "weak_separation"    : "Weak Sep.",
    "medium_separation"  : "Medium Sep.",
    "imbalanced_moderate": r"Imbal.\ Moderate",
    "imbalanced_severe"  : r"Imbal.\ Severe",
    "worst_case"         : "Worst Case",
}

def fmt_val(v, bold=False):
    s = f"{v:.3f}"
    return rf"\textbf{{{s}}}" if bold else s


def build_latex_table(df):
    lines = []
    lines.append(
        r"""\begin{table}[!htbp]
\centering
\footnotesize
\begin{tabular}{ll l ccc ccc}
\toprule
\textbf{Scenario} & \textbf{Frailty Int.} & \textbf{Method}
    & \multicolumn{3}{c}{\textbf{Accuracy}}
    & \multicolumn{3}{c}{\textbf{ARI}} \\\\
\cmidrule(lr){4-6} \cmidrule(lr){7-9}
 & & & Mean & Median & SD & Mean & Median & SD \\\\
\toprule"""
    )

    scenarios = [s for s in SCENARIOS if s in df["Scenario"].unique()]

    for si, scenario in enumerate(scenarios):
        sdf = df[df["Scenario"] == scenario]
        fi_values = sorted(sdf["frailty_intensity"].unique())
        n_rows_scenario = len(sdf)          # total rows for this scenario

        scen_label = SCENARIO_LABELS.get(scenario, scenario)

        if si > 0:
            lines.append(r"\midrule")

        first_scen_row = True

        for fi_i, fi in enumerate(fi_values):
            fdf = sdf[sdf["frailty_intensity"] == fi]
            n_rows_fi = len(fdf)            # rows for this frailty-intensity block

            fi_label = rf"$\gamma={fi:g}$"

            if fi_i > 0:
                lines.append(r"  \cmidrule(lr){2-9}")

            for ri, (_, row) in enumerate(fdf.iterrows()):
                b = row["bold"]
                method_label = METHOD_LABELS[row["Method"]]

                # Column 1: scenario multirow (only on very first row of scenario)
                if first_scen_row and ri == 0:
                    col1 = rf"\multirow{{{n_rows_scenario}}}{{*}}{{{scen_label}}}"
                else:
                    col1 = ""

                # Column 2: frailty-intensity multirow (first row of each block)
                if ri == 0:
                    col2 = rf"  & \multirow{{{n_rows_fi}}}{{*}}{{{fi_label}}}"
                else:
                    col2 = "  &"

                data_cols = " & ".join([
                    fmt_val(row["Acc_mean"],   b),
                    fmt_val(row["Acc_median"], b),
                    fmt_val(row["Acc_sd"],     b),
                    fmt_val(row["ARI_mean"],   b),
                    fmt_val(row["ARI_median"], b),
                    fmt_val(row["ARI_sd"],     b),
                ])

                line = f"  {col1}{col2} & {method_label} & {data_cols}\\\\"
                lines.append(line)
                first_scen_row = False

    lines.append(
        r"""\bottomrule
\end{tabular}
\caption{\small Empirical means, medians, and standard deviations of \textit{accuracy} and \textit{ARI} over $B=100$ replications for the two-step baseline method (K-Means, Hierarchical, Spectral clustering), evaluated across scenarios and frailty intensities. The best method in each setting is highlighted in bold.}
\label{tab:twostep_AccuracyARI}
\end{table}"""
    )

    return "\n".join(lines)


latex_str = build_latex_table(summary_df)
print(latex_str)


\begin{table}[!htbp]
\centering
\footnotesize
\begin{tabular}{ll l ccc ccc}
\toprule
\textbf{Scenario} & \textbf{Frailty Int.} & \textbf{Method}
    & \multicolumn{3}{c}{\textbf{Accuracy}}
    & \multicolumn{3}{c}{\textbf{ARI}} \\\\
\cmidrule(lr){4-6} \cmidrule(lr){7-9}
 & & & Mean & Median & SD & Mean & Median & SD \\\\
\toprule
  \multirow{6}{*}{Baseline}  & \multirow{2}{*}{$\gamma=0.5$} & Hierarchical (Single) & 0.372 & 0.358 & 0.061 & 0.020 & 0.000 & 0.096\\
    & & Hierarchical (Ward) & \textbf{0.904} & \textbf{0.910} & \textbf{0.037} & \textbf{0.747} & \textbf{0.754} & \textbf{0.078}\\
  \cmidrule(lr){2-9}
    & \multirow{2}{*}{$\gamma=1$} & Hierarchical (Single) & 0.368 & 0.358 & 0.053 & 0.015 & 0.000 & 0.082\\
    & & Hierarchical (Ward) & \textbf{0.907} & \textbf{0.914} & \textbf{0.032} & \textbf{0.753} & \textbf{0.760} & \textbf{0.073}\\
  \cmidrule(lr){2-9}
    & \multirow{2}{*}{$\gamma=1.5$} & Hierarchical (Single) & 0.365 & 0.357 & 0.045 & 0.009 & 0.000 & 0.065\\
    & & H

In [ ]:
# ── Save LaTeX table to file ───────────────────────────────────────────────
out_path = Path("TabS62_AccuracyARI.tex")
out_path.write_text(latex_str)
print(f"LaTeX table written to {out_path.resolve()}")

LaTeX table written to /Users/alessandragni/Documents/DATA/POLITECNICO/PHD/CODE_REPO/HazClust/simulations/TabS63_AccuracyARI.tex
